# 🎬 AI Text-to-Image & Video Studio - Google Colab GPU Backend

Notebook này khởi chạy **FastAPI Backend + Z-Image-Turbo (GGUF Quantization) + Motion Engine 3D** trên GPU miễn phí của Google Colab.

### ⚡ Quy trình 3 bước đơn giản:
1. **Bước 1: Cài đặt Môi trường**: Cài đặt PyTorch GPU, Diffusers, GGUF, BitsAndBytes, Cloudflare Tunnel.
2. **Bước 2: Tải trước Toàn bộ Model AI về Colab**: Tải trực tiếp file DiT GGUF và weights Qwen 4-bit về thư mục `/content/models` trên Colab.
3. **Bước 3: Khởi chạy API Backend & Cloudflare Tunnel**: Khởi động máy chủ API siêu tốc và nhận liên kết công khai `https://xxx.trycloudflare.com` kết nối với ứng dụng Client GUI trên máy tính.

---
### 📋 Hướng dẫn sử dụng:
- Vào menu **Runtime** ➔ **Change runtime type** ➔ Chọn **T4 GPU** (hoặc A100 GPU).
- Nhấn **Run all (Chạy tất cả)** hoặc chạy lần lượt 3 ô code bên dưới.
- Copy link Cloudflare dán vào ô **Server API URL** trên Client GUI.

In [ ]:
#@title 1. Cài đặt Môi trường & Thư viện (PyTorch, Diffusers, GGUF, Cloudflared)
import os
import sys

!nvidia-smi

print("\n--- 1. Chuẩn bị mã nguồn dự án... ---")
if not os.path.exists("/content/imagetovideo"):
    !git clone https://github.com/akavipno01/imagetovideo.git /content/imagetovideo
else:
    %cd /content/imagetovideo
    !git pull

%cd /content/imagetovideo

print("\n--- 2. Cài đặt các thư viện AI (Accelerate, GGUF, BitsAndBytes, Diffusers)... ---")
!pip install -q accelerate gguf sentencepiece protobuf huggingface_hub
!pip install -q --upgrade git+https://github.com/huggingface/diffusers
!pip install -q -U bitsandbytes transformers
!pip install -q -r backend/requirements.txt

print("\n--- 3. Tải Cloudflare Tunnel (cloudflared)... ---")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

print("\n✅ Bước 1 hoàn tất! Môi trường GPU đã sẵn sàng.")

In [ ]:
#@title 2. Tải trước Model DiT GGUF về Colab (Z-Image-Turbo GGUF)
import os
from huggingface_hub import hf_hub_download

models_dir = "/content/models"
os.makedirs(models_dir, exist_ok=True)

print("=================================================================")
print("📥 BẮT ĐẦU TẢI TRƯỚC MODEL TRỰC TIẾP TRÊN GOOGLE COLAB")
print("=================================================================\n")

# 1. Tải Transformer DiT file GGUF Q4_K_M từ Unsloth
print("1️⃣ Đang tải file DiT Transformer GGUF (z-image-turbo-Q4_K_M.gguf)...")
gguf_path = hf_hub_download(
    repo_id="unsloth/Z-Image-Turbo-GGUF",
    filename="z-image-turbo-Q4_K_M.gguf",
    local_dir=models_dir,
)
print(f"   ✅ Đã tải xong GGUF tại: {gguf_path}\n")

print("="*65)
print("🎉 MODEL GGUF ĐÃ ĐƯỢC TẢI VỀ Ổ CỨNG COLAB HOÀN TẤT!")
print(f"📁 Thư mục lưu trữ: {models_dir}")
print("="*65)

In [ ]:
#@title 3. Khởi chạy FastAPI Backend & Cloudflare Tunnel
import subprocess
import time
import re
import os

data_dir = "/content/data"
os.makedirs(os.path.join(data_dir, "outputs", "images"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "outputs", "videos"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "temp"), exist_ok=True)
os.makedirs(os.path.join(data_dir, "models"), exist_ok=True)

print("🚀 Đang khởi động Text-to-Image & Video FastAPI Backend (Model đã sẵn sàng tại local)...\n")
backend_process = subprocess.Popen(
    ["python", "run.py"],
    cwd="/content/imagetovideo/backend",
    env={
        **os.environ,
        "TEXT_TO_VIDEO_PORT": "3930",
        "TEXT_TO_VIDEO_DATA_DIR": data_dir,
        "Z_IMAGE_MODEL_PATH": "/content/models/Z-Image-Turbo",
        "Z_IMAGE_GGUF_PATH": "/content/models/z-image-turbo-Q4_K_M.gguf",
    }
)

time.sleep(3)

print("🌐 Đang khởi tạo đường truyền kết nối công khai qua Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:3930"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

colab_url = None
try:
    while True:
        line = tunnel_process.stdout.readline()
        if not line:
            break
        if "trycloudflare.com" in line or "error" in line.lower() or "tunnel" in line.lower():
            print("[Cloudflared]", line.strip())
        
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            colab_url = match.group(0)
            print("\n" + "="*70)
            print(f"🎉 KHỞI CHẠY GOOGLE COLAB AI VIDEO BACKEND THÀNH CÔNG!")
            print(f"🔗 Địa chỉ Server API URL công khai của bạn là:")
            print(f"   >>>  {colab_url}  <<<")
            print("="*70)
            print("\n💡 HƯỚNG DẪN KẾT NỐI CLIENT GUI:")
            print(f"1. Mở ứng dụng Client GUI trên máy tính.")
            print(f"2. Dán link trên vào ô 'Server API URL' rồi bấm 'Kiểm Tra Kết Nối'.")
            print(f"3. Nhập Prompt và bắt đầu sinh Ảnh / Video tốc độ cao!")
            print("="*70 + "\n")
            
    backend_process.wait()
except KeyboardInterrupt:
    print("\nĐang dừng hệ thống...")
    tunnel_process.terminate()
    backend_process.terminate()
    print("Đã dừng.")
